# Routing Scenario Analysis

- physical scenario
  - scenarios
    - fwd / bwd
    - hazard / no hazard
    - ablation scenario
  - plot
    - routes by month for physical scenario
    - elite_cost vs. elite_length for physical scenario

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

import pandas as pd
import geopandas as gpd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import cartopy 

from load_tuning_results import (
    load_results_raw,
    load_result_for_key,
    add_derived_features,
    filter_suspicious_routes,
)

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
gdf = gpd.read_parquet("../results/results_prelim.geoparquet")
gdf = add_derived_features(gdf)
gdf

In [ ]:
gdf = filter_suspicious_routes(gdf)

In [ ]:
gdf.columns

## Cost Distributions

In [ ]:
def plot_cost_distributions(
    gdf: gpd.GeoDataFrame = None,
    forcing_scenario_name: str = "baseline",
    hyper_hazard_penalty_multiplier: float = 0.0,
    relabs: str = "absolute",
):
    _gdf = gdf.where(
        (gdf.hyper_hazard_penalty_multiplier == hyper_hazard_penalty_multiplier)
        & (gdf.forcing_scenario_name == forcing_scenario_name)
    ).dropna(how="all")
    _gdf = _gdf.assign(
        time_start=pd.to_datetime(_gdf["journey_time_start"]).dt.strftime("%Y-%m")
    )

    g = sns.catplot(
        data=_gdf,
        x=f"elite_cost_{relabs}",
        row="journey_name",
        col="journey_speed_knots",
        hue="time_start",
        kind="violin",
        height=3,
        aspect=2,
        inner="quartile",
        legend=True,
        margin_titles=True,
    )
    g.set_titles(col_template=f"{{col_name}} knots, {forcing_scenario_name}", row_template="{row_name}")
    g.fig.savefig(f"../figures/02_cost_distribution_{relabs}_scen-{forcing_scenario_name}_hazardmult-{hyper_hazard_penalty_multiplier}.png", dpi=200)
    g.fig.savefig(f"../figures/02_cost_distribution_{relabs}_scen-{forcing_scenario_name}_hazardmult-{hyper_hazard_penalty_multiplier}.pdf", dpi=200)
    return g.fig

In [ ]:
for fsn in gdf.forcing_scenario_name.unique():
    for relabs in ["absolute", "relative"]:
        plot_cost_distributions(gdf=gdf, forcing_scenario_name=fsn, hyper_hazard_penalty_multiplier=0, relabs=relabs);

## Best routes per month

In [ ]:
def plot_routes_per_month(
    gdf: gpd.GeoDataFrame = None,
    forcing_scenario_name: str = "baseline",
    hyper_hazard_penalty_multiplier: float = 0,
):
    _gdf = gdf.where(
        (gdf.hyper_hazard_penalty_multiplier == hyper_hazard_penalty_multiplier)
        & (gdf.forcing_scenario_name == forcing_scenario_name)
    ).dropna(how="all")
    _gdf = _gdf.assign(
        time_start=pd.to_datetime(_gdf["journey_time_start"]).dt.strftime("%Y-%m")
    )
    cols_scenario = ['journey_name', 'journey_speed_knots', 'journey_time_start']
    _gdf_best = _gdf.sort_values(by="elite_cost_relative").groupby(cols_scenario).first().reset_index()
    
    fig, ax = plt.subplots(
        3, 2, 
        subplot_kw={
            "projection": cartopy.crs.Stereographic(
                central_latitude=_gdf.geometry.centroid.y.mean(), 
                central_longitude=_gdf.geometry.centroid.x.mean(),
            ),
        },
        sharex=True, sharey=True,
        figsize=(8, 7),
    )
    for n, jsk in enumerate(sorted(_gdf.journey_speed_knots.unique())):
        for m, jn in enumerate(_gdf.journey_name.unique()):
            _ax = ax[n, m]
            _gdf_best.where(
                (_gdf_best.journey_name == jn) & (_gdf_best.journey_speed_knots == jsk)
            ).plot(
                column="journey_time_start",
                # cmap="viridis",
                legend=False,
                ax=_ax,
                transform=cartopy.crs.PlateCarree()
            )
            hdl, _ = _ax.get_legend_handles_labels()
            _ax.coastlines()
            _ax.gridlines()
            _ax.set_title(f"{jn:s}: {jsk:.1f} knots")
    fig.suptitle(f"Best routes per month, {forcing_scenario_name}")
    fig.tight_layout()
    fig.savefig(f"../figures/02_best_routes_scen-{forcing_scenario_name}.png", dpi=200)
    fig.savefig(f"../figures/02_best_routes_scen-{forcing_scenario_name}.pdf", dpi=200)
    return fig

In [ ]:
for fsn in gdf.forcing_scenario_name.unique():
    plot_routes_per_month(gdf=gdf, forcing_scenario_name=fsn, hyper_hazard_penalty_multiplier=0);